## Execution Example
This notebook shows the timing side-channel break against `verify_login` (`timing_compare`). The comparison returns at the first mismatched byte, so its duration leaks how long a prefix matched: the correct byte runs one more amplified iteration before exiting. We reuse the week-4 studio attack (`timing_attack` with `time_guesses` from `studios/week-04/rsa_lab.py`), which interleaves the 256 candidate bytes so CPU drift cannot bias the candidates measured last.

In [1]:
from verify_login import timing_attack
from duel1_targets import timing_compare, _TIMING_SECRET

In [2]:
recovered = timing_attack(len(_TIMING_SECRET), timing_compare, rounds=41)

print("recovered:", recovered.hex())
print("accepted by the deployment's own check:", timing_compare(recovered))
print("matches the hidden secret:", recovered == _TIMING_SECRET)

recovered: 83fabf35
accepted by the deployment's own check: True
matches the hidden secret: True


### Reliability
Timing is noisy, so this is not a one-shot claim: each byte is decided by the median of 41 timed calls, interleaved across all 256 candidates. Below are five independent full recoveries.

In [3]:
runs = 5
wins = 0
for i in range(runs):
    got = timing_attack(len(_TIMING_SECRET), timing_compare, rounds=41)
    ok = got == _TIMING_SECRET
    wins += ok
    print(f"run {i + 1}/{runs}: {got.hex()} {'OK' if ok else 'WRONG'}")

print(f"success rate: {wins}/{runs} with 41 interleaved rounds per candidate")

run 1/5: 83fabf35 OK
run 2/5: 83fabf35 OK
run 3/5: 83fabf35 OK
run 4/5: 83fabf35 OK
run 5/5: 83fabf35 OK
success rate: 5/5 with 41 interleaved rounds per candidate


### The fix
The comparison's result was never wrong -- only its duration leaked. Making it constant-time removes the signal: `constant_time_equal` from the week-4 studio examines every byte and never early-exits (in real code, call `hmac.compare_digest`).

In [4]:
import hmac
from rsa_lab import make_oracle

def constant_time_equal(a, b):
    if len(a) != len(b):
        return False
    acc = 0
    for x, y in zip(a, b):
        acc |= x ^ y
    return acc == 0

fixed_oracle = make_oracle(_TIMING_SECRET, compare=constant_time_equal)
got = timing_attack(len(_TIMING_SECRET), fixed_oracle, rounds=41)
print("same attack against the constant-time compare:", got.hex())
print("recovered nothing:", got != _TIMING_SECRET)
print("stdlib version (hmac.compare_digest):", hmac.compare_digest(_TIMING_SECRET, _TIMING_SECRET))

same attack against the constant-time compare: ad000000
recovered nothing: True
stdlib version (hmac.compare_digest): True


**Answer**: `_TIMING_SECRET = 83fabf35`, recovered byte by byte from timing alone (5/5 full recoveries, 41 interleaved rounds per candidate, ~34 us of separation per byte). The deployment's own check accepts the recovered secret, and the same attack recovers nothing against a constant-time compare.